# Setting Up Databricks Connect

Every code example in this course runs on Databricks serverless compute, but
you write and run it from your own editor. The bridge between the two is
**Databricks Connect**, a client library for the Databricks Runtime that
lets local code connect to Databricks compute. General Python code runs on
your machine, while DataFrame operations run on Databricks compute; under the
hood it is built on open-source Spark Connect
([What is Databricks Connect?](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/#how-does-it-work)).
{doc}`Spark Runtime Architecture <../01_introduction_to_apache_spark/01_apache_spark_runtime_architecture/02_spark_runtime_architecture>`
explains what that split means for your code.

## Requirements

- A Databricks workspace with Unity Catalog and serverless compute enabled;
  serverless compute is supported from Databricks Connect 15.1
  ([Databricks Connect usage requirements](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/requirements#workspace-requirements)).
  Databricks Free Edition gives you such a workspace, with serverless
  compute only
  ([Free Edition limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations)).
- Python 3.12 and the Databricks CLI on your machine
  ([Tutorial: Run Python code on serverless compute](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/python/tutorial-serverless#requirements)).
- A Databricks Connect version no newer than the Databricks Runtime on
  serverless compute
  ([Databricks Connect usage requirements](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/requirements#workspace-requirements)).
  This repository pins `databricks-connect==19.1.*` in `pyproject.toml`,
  matching serverless environment version 6
  ([Serverless environment version 6](https://docs.databricks.com/aws/en/release-notes/serverless/environment-version/six)).

## Authenticate Against a Databricks Workspace

Log in with OAuth from the command line. The CLI opens a browser, then
prompts for a **profile name**, which you will pass to the session later.
This course uses `free-edition`:

```shell
databricks auth login --host <workspace-url>
```

The profile, holding only non-secret settings such as the host, is saved in
`~/.databrickscfg`, and the OAuth token is cached in your operating system's
secure storage. List your profiles with `databricks auth profiles`
([Authentication for the Databricks CLI](https://docs.databricks.com/aws/en/dev-tools/cli/authentication#oauth-user-to-machine-u2m-authentication)).

## Install the Project

Databricks strongly recommends installing Databricks Connect into a virtual
environment, and `pyspark` must not be installed alongside it, because the
two packages conflict
([Install Databricks Connect for Python](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/python/install)).
This repository manages both with `uv`, so from the repository root:

```shell
uv sync
```

## Test Your Setup

The next cell creates a session on serverless compute with the `free-edition`
profile and reads five rows from a sample table, the same check as the
official tutorial
([Tutorial: Run Python code on serverless compute](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/python/tutorial-serverless#step-4-add-code-and-run)):

In [1]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.serverless().profile("free-edition").getOrCreate()

df = spark.read.table("samples.nyctaxi.trips")
df.show(5)

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|
+--------------------+---------------------+-------------+-----------+----------+-----------+
only showing top 5 rows


The course's notebooks use a small wrapper around the same builder,
`project.utils.sparksession.get_spark_session`, which falls back to a local
`SparkSession` if Databricks Connect isn't installed. It should print the
same rows:

In [2]:
from project.utils.sparksession import get_spark_session

spark = get_spark_session(serverless=True, profile="free-edition")

df = spark.read.table("samples.nyctaxi.trips")
df.show(5)

+--------------------+---------------------+-------------+-----------+----------+-----------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|
+--------------------+---------------------+-------------+-----------+----------+-----------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|
| 2016-02-13 18:29:09|  2016-02-13 18:37:23|         1.31|        7.5|     10023|      10023|
| 2016-02-06 19:40:58|  2016-02-06 19:52:32|          1.8|        9.5|     10001|      10018|
| 2016-02-12 19:06:43|  2016-02-12 19:20:54|          2.3|       11.5|     10044|      10111|
| 2016-02-23 10:27:56|  2016-02-23 10:58:33|          2.6|       18.5|     10199|      10022|
+--------------------+---------------------+-------------+-----------+----------+-----------+
only showing top 5 rows


The session is a Spark Connect session, and the server reports its Spark
version:

In [3]:
print(type(spark))
print(spark.version)

<class 'pyspark.sql.connect.session.SparkSession'>


4.2.0


## When to Use Databricks Connect

| Where you run code | Good for |
| --- | --- |
| Databricks Connect from your IDE, as in this course | Writing, debugging and testing code locally with your own tools, against real Databricks compute |
| A notebook in the Databricks workspace | Exploring data interactively, with `display()` and the workspace UI |
| A job | Running finished code on a schedule |

For production code, the tutorial recommends leaving the compute out of the
session builder, `DatabricksSession.builder.getOrCreate()`, and configuring
it in the profile instead, for example with `serverless_compute_id = auto`
([Tutorial: Run Python code on serverless compute](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/python/tutorial-serverless#step-5-make-your-code-production-ready)).

## Common Mistakes

- **Having `pyspark` installed next to `databricks-connect`.** The packages
  conflict; uninstall `pyspark` first
  ([Troubleshooting Databricks Connect for Python](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/python/troubleshooting#conflicting-pyspark-installations)).
- **A local Python that doesn't match.** For UDFs, the local minor Python
  version must match the Databricks Runtime's
  ([Databricks Connect usage requirements](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/requirements#local-environment-requirements));
  use Python 3.12.
- **A Databricks Connect version newer than serverless supports.** Keep the
  client at or below the serverless Databricks Runtime release
  ([Databricks Connect usage requirements](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/requirements#workspace-requirements)).
- **A profile name in code that doesn't exist.** The name passed to
  `.profile(...)` must be one that `databricks auth profiles` lists.

With the session working, start the course with
{doc}`Introduction to Apache Spark <../01_introduction_to_apache_spark/index>`.

## Resources

- [Tutorial: Run Python code on serverless compute (Databricks)](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/python/tutorial-serverless)
- [What is Databricks Connect? (Databricks)](https://docs.databricks.com/aws/en/dev-tools/databricks-connect/)